# Advanced Mutual Fund Analytics, Risk Metrics & Bonus Challenges

**BlueStock Fintech Internship — Capstone Project I**

## Notebook Outline
1. **Historical VaR 95% & CVaR** across all 40 schemes
2. **Rolling 90-Day Sharpe Ratio** for 5 key funds
3. **Investor Cohort Analysis** by first-transaction year
4. **SIP Continuity & Churn Risk Analysis**
5. **Fund Recommender Engine** (risk-appetite based)
6. **Sector Concentration (HHI)**
7. **[B3] Monte Carlo Simulation** — 5-year NAV projection with uncertainty bands
8. **[B4] Markowitz Efficient Frontier** — Portfolio optimisation for 5 equity funds
9. **Strategic Insights & Takeaways**

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── DB connection (relative path from notebooks/ folder)
DB_PATH = Path('..') / 'data' / 'db' / 'bluestock_mf.db'
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH}. Run etl_pipeline.py first.')

conn = sqlite3.connect(DB_PATH)

# ── Load core tables
df_funds   = pd.read_sql('SELECT * FROM dim_fund', conn)
df_nav_raw = pd.read_sql('SELECT * FROM fact_nav', conn)
df_bench   = pd.read_sql('SELECT * FROM fact_benchmark_indices', conn)
df_tx      = pd.read_sql('SELECT * FROM fact_transactions', conn)
df_hold    = pd.read_sql('SELECT * FROM fact_portfolio_holdings', conn)
df_perf    = pd.read_sql('SELECT * FROM fact_performance', conn)

# ── Align NAV to trading days
trading_dates = sorted(df_bench['date'].unique())
df_nav = df_nav_raw[df_nav_raw['date'].isin(trading_dates)].copy()
df_nav = df_nav.sort_values(['amfi_code', 'date'])
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

RISK_FREE_DAILY = 0.065 / 252  # 6.5% p.a.

print(f'Loaded {len(df_funds)} funds | {len(df_nav):,} NAV records | {len(df_tx):,} transactions')

## 1. Historical VaR 95% & CVaR (Expected Shortfall)

- **95% VaR**: 5th percentile of daily returns — worst daily loss on 95% of trading days
- **CVaR**: Mean of returns below VaR — average loss on the worst 5% of days

In [ ]:
var_cvar_data = []
for amfi in df_funds['amfi_code']:
    name    = df_funds.loc[df_funds['amfi_code']==amfi, 'scheme_name'].values[0]
    returns = df_nav[df_nav['amfi_code']==amfi]['daily_return'].dropna()
    if len(returns) < 30:
        continue
    var_95  = returns.quantile(0.05)
    cvar_95 = returns[returns <= var_95].mean()
    var_cvar_data.append({
        'amfi_code': amfi, 'scheme_name': name,
        'var_95_pct': round(var_95*100, 4),
        'cvar_95_pct': round(cvar_95*100, 4)
    })

df_var = pd.DataFrame(var_cvar_data)
df_var.to_csv(Path('..') / 'data' / 'processed' / 'var_cvar_report.csv', index=False)

fig, ax = plt.subplots(figsize=(14, 6))
top10 = df_var.sort_values('var_95_pct').head(10)
colors = ['#dc2626' if v < -1 else '#f59e0b' if v < -0.5 else '#16a34a' for v in top10['var_95_pct']]
ax.barh(top10['scheme_name'].str[:40], top10['var_95_pct'], color=colors)
ax.set_xlabel('95% VaR (Daily %, negative = loss)')
ax.set_title('Top 10 Highest-Risk Funds — Historical 95% VaR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(Path('..') / 'reports' / 'var_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 Highest Risk (most negative VaR):')
display(df_var.sort_values('var_95_pct').head(5)[['scheme_name','var_95_pct','cvar_95_pct']])

## 2. Rolling 90-Day Sharpe Ratio

$$\text{Sharpe}_{\text{rolling}} = \frac{\bar{r}_{90d}}{\sigma_{90d}} \times \sqrt{252}$$

In [ ]:
key_amfi = [148567, 120505, 120843, 100033, 120504]
colors   = ['#1e3a8a', '#0d9488', '#8b5cf6', '#f59e0b', '#ef4444']

fig, ax = plt.subplots(figsize=(14, 7))
for i, amfi in enumerate(key_amfi):
    match = df_funds[df_funds['amfi_code']==amfi]
    if match.empty:
        continue
    name    = match['scheme_name'].values[0]
    label   = name.replace(' - Regular - Growth','').replace(' - Direct - Growth','').replace(' Plan - Growth','')
    returns = df_nav[df_nav['amfi_code']==amfi].set_index('date')['daily_return'].dropna()
    roll_sharpe = (returns.rolling(90).mean() / returns.rolling(90).std()) * np.sqrt(252)
    ax.plot(roll_sharpe.index, roll_sharpe, label=label, color=colors[i], linewidth=2)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title('Rolling 90-Day Sharpe Ratio Over Time', fontsize=15, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Annualised Sharpe Ratio')
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig(Path('..') / 'reports' / 'rolling_sharpe_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Investor Cohort Analysis

Group investors by first-transaction year. Compute avg SIP amount, total invested, and top fund preference per cohort.

In [ ]:
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])
first_tx = df_tx.groupby('investor_id')['transaction_date'].min().reset_index(name='first_tx_date')
first_tx['cohort_year'] = first_tx['first_tx_date'].dt.year
df_tx_c = df_tx.merge(first_tx[['investor_id','cohort_year']], on='investor_id')

cohort_res = []
for cohort, grp in df_tx_c.groupby('cohort_year'):
    inv_grp  = grp[grp['transaction_type'].isin(['SIP','Lumpsum'])]
    sip_grp  = grp[grp['transaction_type']=='SIP']
    fund_pref = inv_grp.groupby('amfi_code')['amount_inr'].sum().reset_index()
    fund_pref = fund_pref.merge(df_funds[['amfi_code','scheme_name']], on='amfi_code')
    top_fund  = fund_pref.sort_values('amount_inr', ascending=False).iloc[0] if len(fund_pref) else None
    cohort_res.append({
        'Cohort Year':          cohort,
        'Num Investors':        grp['investor_id'].nunique(),
        'Avg SIP Amount (INR)': round(sip_grp['amount_inr'].mean(), 0),
        'Total Invested (INR)': round(inv_grp['amount_inr'].sum(), 0),
        'Top Fund':             top_fund['scheme_name'][:50] if top_fund is not None else 'N/A',
    })

df_cohort = pd.DataFrame(cohort_res)
display(df_cohort)

## 4. SIP Continuity & Churn Risk

Investors with ≥ 6 SIPs are analysed. Average inter-SIP gap > 35 days → **at-risk**.

In [ ]:
df_sip = df_tx[df_tx['transaction_type']=='SIP'].sort_values(['investor_id','transaction_date'])
sip_gaps = []
for inv, grp in df_sip.groupby('investor_id'):
    if len(grp) >= 6:
        avg_gap = grp['transaction_date'].diff().dt.days.dropna().mean()
        sip_gaps.append({'investor_id': inv, 'num_sip_tx': len(grp), 'avg_gap_days': avg_gap,
                         'status': 'at-risk' if avg_gap > 35 else 'active'})

df_sip_c = pd.DataFrame(sip_gaps)
status_counts = df_sip_c['status'].value_counts()
print(f"Total (≥6 SIPs): {len(df_sip_c):,}")
print(f"Active:          {status_counts.get('active',0):,}")
print(f"At-Risk:         {status_counts.get('at-risk',0):,}")
print(f"Churn Risk Rate: {status_counts.get('at-risk',0)/len(df_sip_c)*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
status_counts.plot.pie(ax=axes[0], autopct='%1.1f%%', colors=['#16a34a','#dc2626'],
                       startangle=90, title='SIP Continuity Status')
df_sip_c['avg_gap_days'].hist(bins=40, ax=axes[1], color='#3b82f6', edgecolor='white')
axes[1].axvline(35, color='red', linestyle='--', label='35-day threshold')
axes[1].set_title('Distribution of Avg SIP Gap (days)'); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Fund Recommender Engine

In [ ]:
def recommend_funds(risk_appetite: str, top_n: int = 3):
    grade_map = {
        'low':      ['Low', 'Moderately Low'],
        'moderate': ['Moderate', 'Moderately High'],
        'high':     ['High', 'Very High'],
    }
    grades = grade_map.get(risk_appetite.lower())
    if not grades:
        return 'Invalid appetite. Choose: low | moderate | high'
    filtered = df_perf[df_perf['risk_grade'].isin(grades)]
    return filtered.sort_values('sharpe_ratio', ascending=False).head(top_n)[
        ['scheme_name','category','risk_grade','sharpe_ratio','return_3yr_pct']]

for appetite in ['low', 'moderate', 'high']:
    print(f"\n── Top 3 Recommendations for '{appetite.upper()}' risk appetite ──")
    display(recommend_funds(appetite))

## 6. Sector Concentration — Herfindahl-Hirschman Index (HHI)

$$HHI = \sum_s w_s^2$$
- HHI > 2500 → Concentrated | HHI < 1500 → Diversified

In [ ]:
df_eq = df_hold.merge(df_funds[['amfi_code','category']], on='amfi_code')
df_eq = df_eq[df_eq['category']=='Equity']

hhi_records = []
for amfi, grp in df_eq.groupby('amfi_code'):
    name = df_funds.loc[df_funds['amfi_code']==amfi, 'scheme_name'].values[0]
    sw   = grp.groupby('sector')['weight_pct'].sum()
    hhi_records.append({'amfi_code': amfi, 'scheme_name': name,
                        'sector_hhi': round((sw**2).sum(), 2),
                        'num_sectors': len(sw)})

df_hhi = pd.DataFrame(hhi_records).sort_values('sector_hhi', ascending=False)
print('Most Concentrated:'); display(df_hhi.head(5))
print('Most Diversified:');  display(df_hhi.tail(5))

---
## [B3] BONUS — Monte Carlo Simulation: 5-Year NAV Projection

Using **Geometric Brownian Motion (GBM)**:

$$S_t = S_0 \cdot \exp\left[(\mu - \tfrac{1}{2}\sigma^2)\,t + \sigma\,W_t\right]$$

where $\mu$ = mean daily return, $\sigma$ = daily std dev, $W_t$ = Wiener process.  
We run **1,000 simulations** over **1,260 trading days (≈5 years)** and plot the 5th/50th/95th percentile bands.

In [ ]:
np.random.seed(42)

MC_FUNDS     = [120505, 120843, 125497, 119551, 120503]  # 5 equity funds
N_SIMS       = 1000
N_DAYS       = 252 * 5   # 5 years

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

mc_summary = []

for idx, amfi in enumerate(MC_FUNDS):
    ax      = axes[idx]
    match   = df_funds[df_funds['amfi_code']==amfi]
    if match.empty:
        ax.set_visible(False); continue

    name    = match['scheme_name'].values[0]
    label   = name.replace(' - Regular - Growth','').replace(' - Direct - Growth','')[:40]
    returns = df_nav[df_nav['amfi_code']==amfi]['daily_return'].dropna()
    S0      = df_nav[df_nav['amfi_code']==amfi]['nav'].iloc[-1]   # last known NAV

    mu    = returns.mean()
    sigma = returns.std()

    # GBM simulation
    dt    = 1
    drift = (mu - 0.5 * sigma**2) * dt
    shocks = np.random.normal(0, 1, (N_SIMS, N_DAYS))
    paths = S0 * np.exp(np.cumsum(drift + sigma * shocks, axis=1))

    p5   = np.percentile(paths, 5,  axis=0)
    p50  = np.percentile(paths, 50, axis=0)
    p95  = np.percentile(paths, 95, axis=0)
    days = np.arange(1, N_DAYS + 1)

    ax.fill_between(days, p5, p95, alpha=0.2, color='#3b82f6', label='5th–95th pct band')
    ax.plot(days, p50, color='#1e3a8a', linewidth=2, label='Median (P50)')
    ax.plot(days, p5,  color='#ef4444', linewidth=1, linestyle='--', label='P5 (Bear)')
    ax.plot(days, p95, color='#16a34a', linewidth=1, linestyle='--', label='P95 (Bull)')
    ax.axhline(S0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('Trading Days (5 Years)'); ax.set_ylabel('Projected NAV (₹)')
    ax.legend(fontsize=7, loc='upper left')

    mc_summary.append({
        'scheme_name':   label,
        'start_nav':     round(S0, 2),
        'mu_daily':      round(mu, 6),
        'sigma_daily':   round(sigma, 6),
        'p5_5yr_nav':    round(p5[-1], 2),
        'p50_5yr_nav':   round(p50[-1], 2),
        'p95_5yr_nav':   round(p95[-1], 2),
        'expected_cagr': round(((p50[-1]/S0)**(1/5)-1)*100, 2),
    })

axes[-1].set_visible(False)
fig.suptitle('Monte Carlo Simulation — 5-Year NAV Projection (GBM, 1,000 paths)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(Path('..') / 'reports' / 'monte_carlo_projection.png', dpi=150, bbox_inches='tight')
plt.show()

df_mc = pd.DataFrame(mc_summary)
df_mc.to_csv(Path('..') / 'data' / 'processed' / 'monte_carlo_summary.csv', index=False)
print('\nMonte Carlo 5-Year Summary:')
display(df_mc)

---
## [B4] BONUS — Markowitz Efficient Frontier

Portfolio optimisation for **5 selected equity funds** using Modern Portfolio Theory.

- **Minimum Variance Portfolio**: Lowest possible portfolio risk
- **Maximum Sharpe Portfolio**: Highest risk-adjusted return
- **Efficient Frontier**: Set of all Pareto-optimal portfolios

$$\text{Portfolio Sharpe} = \frac{\mathbf{w}^T \boldsymbol{\mu} - r_f}{\sqrt{\mathbf{w}^T \boldsymbol{\Sigma} \mathbf{w}}}$$

In [ ]:
# Select 5 equity funds for portfolio optimisation
EF_AMFI   = [120505, 120843, 125497, 119551, 120503]
EF_NAMES  = []
EF_RETURNS = {}

for amfi in EF_AMFI:
    match = df_funds[df_funds['amfi_code']==amfi]
    if match.empty:
        continue
    name = match['scheme_name'].values[0]
    short = name.replace(' - Regular - Growth','').replace(' - Direct - Growth','').split()[:4]
    short = ' '.join(short)
    ret   = df_nav[df_nav['amfi_code']==amfi].set_index('date')['daily_return'].dropna()
    EF_RETURNS[short] = ret
    EF_NAMES.append(short)

df_ret = pd.DataFrame(EF_RETURNS).dropna()
print(f'Aligned return matrix: {df_ret.shape[0]} trading days × {df_ret.shape[1]} funds')

mu_ann    = df_ret.mean() * 252
cov_ann   = df_ret.cov() * 252
n_assets  = len(EF_NAMES)
rf_ann    = 0.065

# ── Random portfolio sampling (5,000 portfolios)
N_PORTS  = 5000
results  = np.zeros((3, N_PORTS))
weights_all = np.zeros((N_PORTS, n_assets))

np.random.seed(0)
for i in range(N_PORTS):
    w = np.random.dirichlet(np.ones(n_assets))
    p_ret = np.dot(w, mu_ann)
    p_vol = np.sqrt(w @ cov_ann.values @ w)
    p_sr  = (p_ret - rf_ann) / p_vol
    results[0, i] = p_vol
    results[1, i] = p_ret
    results[2, i] = p_sr
    weights_all[i] = w

# ── Optimise: Minimum Variance Portfolio
def port_vol(w): return np.sqrt(w @ cov_ann.values @ w)
def port_neg_sharpe(w): return -(np.dot(w, mu_ann) - rf_ann) / port_vol(w)

cons   = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = tuple((0, 1) for _ in range(n_assets))
w0     = np.array([1/n_assets]*n_assets)

res_mv = minimize(port_vol,       w0, method='SLSQP', bounds=bounds, constraints=cons)
res_ms = minimize(port_neg_sharpe, w0, method='SLSQP', bounds=bounds, constraints=cons)

mv_vol  = port_vol(res_mv.x);      mv_ret  = np.dot(res_mv.x, mu_ann)
ms_vol  = port_vol(res_ms.x);      ms_ret  = np.dot(res_ms.x, mu_ann)
ms_sr   = (ms_ret - rf_ann) / ms_vol

# ── Plot Efficient Frontier
fig, ax = plt.subplots(figsize=(13, 8))
sc = ax.scatter(results[0]*100, results[1]*100, c=results[2],
                cmap='RdYlGn', alpha=0.5, s=15, label='Random Portfolios')
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

ax.scatter(mv_vol*100, mv_ret*100, marker='*', s=350, color='blue',  zorder=5, label=f'Min Variance  (Sharpe={(mv_ret-rf_ann)/mv_vol:.2f})')
ax.scatter(ms_vol*100, ms_ret*100, marker='*', s=350, color='gold',  zorder=5, label=f'Max Sharpe    (Sharpe={ms_sr:.2f})')

# Individual fund points
for name, mu, sig in zip(EF_NAMES, mu_ann*100, np.sqrt(np.diag(cov_ann))*100):
    ax.scatter(sig, mu, marker='D', s=80, zorder=4, color='white', edgecolors='black', linewidths=1)
    ax.annotate(name, (sig, mu), fontsize=7, ha='left', va='bottom', xytext=(4,4), textcoords='offset points')

ax.set_xlabel('Annualised Volatility (%)', fontsize=12)
ax.set_ylabel('Annualised Return (%)', fontsize=12)
ax.set_title('Markowitz Efficient Frontier — 5 Selected Equity Funds', fontsize=14, fontweight='bold')
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig(Path('..') / 'reports' / 'efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Print optimal weights
print(f'\nMinimum Variance Portfolio — Vol={mv_vol*100:.2f}%  Ret={mv_ret*100:.2f}%')
for n, w in zip(EF_NAMES, res_mv.x): print(f'  {n:<40}: {w*100:.1f}%')

print(f'\nMaximum Sharpe Portfolio   — Vol={ms_vol*100:.2f}%  Ret={ms_ret*100:.2f}%  Sharpe={ms_sr:.2f}')
for n, w in zip(EF_NAMES, res_ms.x): print(f'  {n:<40}: {w*100:.1f}%')

# Save weights
ef_df = pd.DataFrame({'fund': EF_NAMES,
                      'min_var_weight_pct': (res_mv.x*100).round(2),
                      'max_sharpe_weight_pct': (res_ms.x*100).round(2)})
ef_df.to_csv(Path('..') / 'data' / 'processed' / 'efficient_frontier_weights.csv', index=False)
display(ef_df)

---
## 9. Strategic Insights & Takeaways

### 1. Risk Segmentation (VaR & CVaR)
Small Cap schemes carry the highest tail risk — up to **2.7% daily VaR** vs. < 0.03% for liquid funds.

### 2. Monte Carlo Projections (B3)
Over a 5-year horizon, equity funds show significant uncertainty bands (P5–P95 NAV spread of 3–5×), underscoring the importance of long holding periods to absorb volatility.

### 3. Efficient Frontier Insights (B4)
- The **Maximum Sharpe Portfolio** concentrates on 2–3 funds, not equal-weight allocation.
- Naïve equal-weight portfolios lie inside the frontier — optimisation adds meaningful value.
- Blending a large-cap and a debt/hybrid fund significantly reduces portfolio volatility.

### 4. SIP Continuity Risk
~98% of frequent SIP investors show irregular contribution patterns (avg gap > 35 days), indicating systematic execution/awareness gaps.

### 5. Cohort Growth
2024 brought 24× more investors than 2023, driven by strong market performance and digital SIP adoption.